# COMP8240 – fastText replication (Joulin et al., 2017)
Replicates *Bag of Tricks for Efficient Text Classification* on 8 benchmark datasets, applies it to BBC News, and builds and evaluates BBC-Sentences.

**Run the cells top to bottom in Google Colab (free CPU runtime is enough).** If the runtime restarts, run cells 1 and 2 again before anything else.
Sections 6b, 7 and 8 are self-contained and can be run on their own.


## 1. Build fastText

In [ ]:
%cd /content
![ -d fastText ] || git clone https://github.com/facebookresearch/fastText.git
%cd /content/fastText
!make -j2 2>&1 | tail -1
!./fasttext 2>&1 | head -3      # should print the usage message
!nproc                          # number of CPU threads (record this for the video)

## 2. Helper functions (data prep, training, testing)

In [ ]:
!pip install -q datasets
import re, json, random, os, time, subprocess, pandas as pd
from datasets import load_dataset
from google.colab import files

def norm(t):
    t = (t or "").lower().replace("<br />", " ")
    t = re.sub(r"([.,!?;:()\"'])", r" \1 ", t)
    return re.sub(r"\s+", " ", t).strip()

# name: (Hugging Face ID, text fields, label field)
DATASETS = {
    "ag":    ("fancyzhx/ag_news",         ["text"],             "label"),
    "dbp":   ("fancyzhx/dbpedia_14",      ["title", "content"], "label"),
    "yelpP": ("fancyzhx/yelp_polarity",   ["text"],             "label"),
    "yelpF": ("Yelp/yelp_review_full",    ["text"],             "label"),
    "yahoo": ("community-datasets/yahoo_answers_topics",
              ["question_title", "question_content", "best_answer"], "topic"),
    "amzP":  ("fancyzhx/amazon_polarity", ["title", "content"], "label"),
    "sogou": ("community-datasets/sogou_news", ["title", "content"], "label"),   # pinyin text, 5 classes
}
# Amazon Full is not on the HF Hub in its original form, so it is read from the
# original Zhang et al. (2015) CSV release (mirrored by fast.ai): name -> (url, folder)
CSV_DATASETS = {
    "amzF": ("https://s3.amazonaws.com/fast-ai-nlp/amazon_review_full_csv.tgz", "amazon_review_full_csv"),
}
# Paper, Table 1: (fastText h=10, fastText h=10 + bigrams)
PAPER = {"ag": (91.5, 92.5), "dbp": (98.1, 98.6), "yelpP": (93.8, 95.7),
         "yelpF": (60.4, 63.9), "yahoo": (72.0, 72.3), "amzP": (91.2, 94.6),
         "amzF": (55.8, 60.2), "sogou": (93.9, 96.8)}
THREADS = os.cpu_count()

def prep(name):
    hf, fields, lab = DATASETS[name]
    ds = load_dataset(hf)
    line = lambda ex: f"__label__{ex[lab]+1} " + norm(" ".join(ex[f] or "" for f in fields))
    train_rows = [line(ex) for ex in ds["train"]]
    test_rows  = [line(ex) for ex in ds["test"]]
    random.seed(0); random.shuffle(train_rows)
    cut = int(len(train_rows) * 0.95)          # 5% held out to choose the learning rate
    for fn, rows in [(f"{name}.train", train_rows), (f"{name}.trsub", train_rows[:cut]),
                     (f"{name}.valid", train_rows[cut:]), (f"{name}.test", test_rows)]:
        with open(fn, "w") as f: f.write("\n".join(rows) + "\n")
    print(f"{name}: train {len(train_rows):,}  test {len(test_rows):,}")

def write_splits(name, train_rows, test_rows):
    random.seed(0); random.shuffle(train_rows)
    cut = int(len(train_rows) * 0.95)          # 5% held out to choose the learning rate
    for fn, rows in [(f"{name}.train", train_rows), (f"{name}.trsub", train_rows[:cut]),
                     (f"{name}.valid", train_rows[cut:]), (f"{name}.test", test_rows)]:
        with open(fn, "w") as f: f.write("\n".join(rows) + "\n")
    print(f"{name}: train {len(train_rows):,}  test {len(test_rows):,}")

def prep_csv(name):
    """Download an original Zhang et al. CSV release and write fastText files.
    Columns: class (1-based), then one or more text fields; newlines are escaped as a literal \\n."""
    import csv, sys, tarfile
    csv.field_size_limit(sys.maxsize)
    url, folder = CSV_DATASETS[name]
    if not os.path.exists(folder):
        subprocess.run(f"wget -q -c {url} -O {folder}.tgz", shell=True, check=True)
        tarfile.open(f"{folder}.tgz").extractall(); os.remove(f"{folder}.tgz")
    def read(split):
        with open(f"{folder}/{split}.csv", newline="", encoding="utf-8") as f:
            return [f"__label__{r[0]} " + norm(" ".join(r[1:]).replace("\\n", " "))
                    for r in csv.reader(f)]
    write_splits(name, read("train"), read("test"))

def train(inp, out, ng, lr):
    t = time.time()
    subprocess.run(f"./fasttext supervised -input {inp} -output {out} -dim 10 -epoch 5 "
                   f"-lr {lr} -wordNgrams {ng} -bucket 10000000 -minCount 1 "
                   f"-thread {THREADS} -verbose 0", shell=True, check=True)
    return time.time() - t

def acc(model, data):
    out = subprocess.run(f"./fasttext test {model}.bin {data}", shell=True,
                         capture_output=True, text=True).stdout
    return float(re.search(r"P@1\s+([\d.]+)", out).group(1))

CSV = "replication.csv"
results = pd.read_csv(CSV).to_dict("records") if os.path.exists(CSV) else []

def replicate(name, lrs=(0.05, 0.1, 0.25, 0.5)):
    results[:] = [r for r in results if r["dataset"] != name]   # re-running replaces old rows
    for ng in (1, 2):
        val = {}
        for lr in lrs:
            train(f"{name}.trsub", "tmp", ng, lr)
            val[lr] = acc("tmp", f"{name}.valid")
        best = max(val, key=val.get)
        secs = train(f"{name}.train", f"{name}_{ng}", ng, best)
        ours = round(100 * acc(f"{name}_{ng}", f"{name}.test"), 1)
        paper = PAPER[name][ng - 1]
        results.append({"dataset": name, "bigrams": ng == 2, "lr": best,
                        "ours_%": ours, "paper_%": paper, "diff": round(ours - paper, 1),
                        "train_sec": round(secs, 1), "threads": THREADS})
    for f in ("tmp.bin", "tmp.vec", f"{name}_1.bin"):          # free disk space
        if os.path.exists(f): os.remove(f)
    pd.DataFrame(results).to_csv(CSV, index=False)
    return pd.DataFrame([r for r in results if r["dataset"] == name])

print("Helpers ready, threads =", THREADS)

## 3. Replication on the paper's datasets
Run one cell at a time. Core four: AG, DBpedia, Yelp P, Yelp F.

In [ ]:
# AG News (4 classes)
prep("ag"); replicate("ag")

In [ ]:
# Live demo for the video: train, test, predict
!time ./fasttext supervised -input ag.train -output ag_bi -dim 10 -epoch 5 -lr 0.25 -wordNgrams 2 -bucket 10000000 -minCount 1 -thread 2
!./fasttext test ag_bi.bin ag.test
!echo "stocks rally as tech earnings beat expectations" | ./fasttext predict ag_bi.bin -

In [ ]:
# DBpedia (14 classes)
prep("dbp"); replicate("dbp")

In [ ]:
# Yelp Polarity (positive / negative)
prep("yelpP"); replicate("yelpP")

In [ ]:
# Yelp Full (1-5 stars)
prep("yelpF"); replicate("yelpF")

### Optional (large, slow): only if time allows

In [ ]:
# Yahoo! Answers (10 topics, 1.4M examples): 2 learning rates to save time
prep("yahoo"); replicate("yahoo", lrs=(0.1, 0.25))

In [ ]:
# Amazon Polarity (3.6M examples): 2 learning rates to save time
prep("amzP"); replicate("amzP", lrs=(0.1, 0.25))

### Remaining large datasets: Amazon Full and Sogou
Each cell downloads, preprocesses (same `norm()`, same 95/5 split to choose the learning rate) and runs unigram + bigram.
Keep this tab open while they run: Amazon Full should take about as long as Amazon Polarity (~45 min), Sogou ~15–20 min.
`replication.csv` is saved after each dataset, so a disconnect only loses the one that was running.

In [ ]:
# Amazon Full (1-5 stars, 3.0M train / 650k test): original Zhang et al. CSV, 2 learning rates to save time
prep_csv("amzF"); replicate("amzF", lrs=(0.1, 0.25))

In [ ]:
# Sogou News (5 topics, pinyin, 450k train / 60k test): 2 learning rates to save time
prep("sogou"); replicate("sogou", lrs=(0.1, 0.25))

In [ ]:
# Free disk space once the large runs are done
!rm -f amzF.* sogou.* amzF_2.bin sogou_2.bin
!rm -rf amazon_review_full_csv
!df -h /content | tail -1

In [ ]:
# Results table so far + download
df = pd.DataFrame(results); display(df)
files.download(CSV)

## 4. New existing dataset: BBC News

In [ ]:
bbc = load_dataset("SetFit/bbc-news")          # 5 topics
BBC_NAMES = {ex["label"]: ex["label_text"] for ex in bbc["train"]}
print(BBC_NAMES, "| train", len(bbc["train"]), "| test", len(bbc["test"]))
for split, fn in [("train", "bbc.train"), ("test", "bbc.test")]:
    with open(fn, "w") as f:
        for ex in bbc[split]:
            f.write(f"__label__{ex['label']+1} {norm(ex['text'])}\n")
bbc_res = []
for ng in (1, 2):
    secs = train("bbc.train", f"bbc_{ng}", ng, 0.25)
    bbc_res.append({"data": "BBC articles", "bigrams": ng == 2,
                    "acc_%": round(100 * acc(f"bbc_{ng}", "bbc.test"), 1), "train_sec": round(secs, 1)})
display(pd.DataFrame(bbc_res))

## 5. Constructed dataset: BBC sentences (labels inherited from the article)
Each **raw** article is split into sentences first and each sentence is normalised afterwards.
(The first version split *after* `norm()`, which had already turned `£1.6bn` into `1 . 6bn`, so the splitter
cut numbers in half and produced fragments such as "6bn ) of government bonds".)
Sentences shorter than 5 words are dropped.

In [ ]:
# needs the BBC data from section 4; load it here if section 4 has not been run in this session
if "bbc" not in globals():
    bbc = load_dataset("SetFit/bbc-news")
    BBC_NAMES = {ex["label"]: ex["label_text"] for ex in bbc["train"]}
bbc_res = globals().get("bbc_res", [])

# a sentence ends with . ! or ? followed by whitespace; "u.s. economy" style abbreviations are not split
SENT_SPLIT = re.compile(r"(?<![a-z]\.[a-z]\.)(?<=[.!?])\s+")

def looks_fragment(s):
    """Heuristic: a sentence that starts with a number, bracket or currency sign was probably cut mid-sentence."""
    return bool(re.match(r"^[\d)\]$£€%.,]", s))

def fragment_rate(fn):
    lines = [l.split(" ", 1)[1] for l in open(fn).read().splitlines()]
    return sum(map(looks_fragment, lines)), len(lines)

old = fragment_rate("bbcsent.test") if os.path.exists("bbcsent.test") else None

def to_sentences(split, dst, min_words=5):
    n = 0
    with open(dst, "w") as out:
        for ex in bbc[split]:
            for s in SENT_SPLIT.split(ex["text"]):
                s = norm(s)
                if len(s.split()) >= min_words:
                    out.write(f"__label__{ex['label']+1} {s}\n"); n += 1
    return n

print("sentences: train", to_sentences("train", "bbcsent.train"), "| test", to_sentences("test", "bbcsent.test"))
new = fragment_rate("bbcsent.test")
if old: print(f"likely fragments in test set: before fix {old[0]}/{old[1]} ({100*old[0]/old[1]:.1f}%)", end=" -> ")
print(f"after fix {new[0]}/{new[1]} ({100*new[0]/new[1]:.1f}%)")

bbc_res = [r for r in bbc_res if r["data"] != "BBC sentences"]      # re-running replaces old rows
for ng in (1, 2):
    secs = train("bbcsent.train", f"bbcsent_{ng}", ng, 0.25)
    bbc_res.append({"data": "BBC sentences", "bigrams": ng == 2,
                    "acc_%": round(100 * acc(f"bbcsent_{ng}", "bbcsent.test"), 1), "train_sec": round(secs, 1)})
display(pd.DataFrame(bbc_res))
pd.DataFrame(bbc_res).to_csv("bbc_results.csv", index=False)

## 6. Annotation sheet (hand-label in Excel / Google Sheets)
This exports **two files**:
* `bbc_annotation_sheet.csv` – what annotators see: sentence + the article's topic, **no fastText prediction** (so it cannot bias them).
* `bbc_annotation_key.csv` – the same 100 sentences with fastText's prediction, kept for the analysis in section 8.

Each annotator fills in their own copy of the sheet **independently** (don't look at each other's answers), then saves it as `annotator_A.csv` / `annotator_B.csv`:
* `on_topic`: **Y** if, reading only this sentence, you would assign it to the given topic; **N** if not.
* `true_topic`: the topic the sentence on its own signals – `business`, `entertainment`, `politics`, `sport`, `tech` – or `none`.

In [ ]:
assert os.path.exists("bbcsent_2.bin"), "Run section 5 first: it builds the sentence data and the bigram model used here."
lines = open("bbcsent.test").read().splitlines()
random.seed(42); sample = random.sample(lines, 100)
labels, texts = zip(*(l.split(" ", 1) for l in sample))
open("sample.txt", "w").write("\n".join(texts) + "\n")
preds = subprocess.run("./fasttext predict bbcsent_2.bin sample.txt", shell=True,
                       capture_output=True, text=True).stdout.split()
name = lambda lab: BBC_NAMES[int(lab.replace("__label__", "")) - 1]

key = pd.DataFrame({"id": range(100), "sentence": texts,
                    "inherited_label": [name(l) for l in labels],
                    "fasttext_pred": [name(p) for p in preds]})
sheet = key[["id", "sentence", "inherited_label"]].assign(on_topic="", true_topic="")
key.to_csv("bbc_annotation_key.csv", index=False)
sheet.to_csv("bbc_annotation_sheet.csv", index=False)
print("fastText agrees with inherited label:", (key.inherited_label == key.fasttext_pred).sum(), "/ 100")
print("likely fragments in the sample:", sum(map(looks_fragment, texts)), "/ 100")
files.download("bbc_annotation_sheet.csv"); files.download("bbc_annotation_key.csv"); files.download("bbc_results.csv")

### 6b. Annotation tool and guidelines
Run the cell below and click one topic per sentence; it saves `annotator_A.csv` after every click and resumes where you stopped.
The annotator sees only the sentence (not the article's topic or fastText's prediction). `on_topic` is filled in automatically: **Y** if the chosen topic equals the article's topic.

Guidelines (both annotators use the same rules):
1. Judge the sentence **on its own** – do not guess what the rest of the article was about.
2. Pick the topic the sentence is **mainly** about; general knowledge a UK news reader would have counts (e.g. a footballer's name signals sport).
3. Pick **none** if the sentence gives no topic clue (e.g. "those were games we lost" is sport, but "he s a decent lad" is none).
4. Don't go back to change earlier answers after seeing later sentences unless you clicked by mistake.

The second annotator uses the same cell with `ANNOTATOR = "B"`, ideally in their own copy of the notebook, without seeing `annotator_A.csv`.

In [ ]:
# ---- Annotation tool: one sentence at a time, saved after every click ----
# Self-contained: works after a runtime restart. The second annotator changes ANNOTATOR to "B".
import os
import pandas as pd
import ipywidgets as w
from IPython.display import display
from google.colab import files

ANNOTATOR = "A"
SHOW_LABEL = False      # keep False: annotators label blind to the article's topic and to fastText
TOPICS = ["business", "entertainment", "politics", "sport", "tech", "none"]

os.makedirs("/content/fastText", exist_ok=True); os.chdir("/content/fastText")
if not os.path.exists("bbc_annotation_sheet.csv"):
    print("Please upload bbc_annotation_sheet.csv"); files.upload()
OUT = f"annotator_{ANNOTATOR}.csv"
src = OUT if os.path.exists(OUT) else "bbc_annotation_sheet.csv"     # resume if you stopped part-way
ann = pd.read_csv(src, dtype=str, keep_default_na=False)
for col in ("on_topic", "true_topic"):
    if col not in ann: ann[col] = ""

def next_unlabelled(start=0):
    for i in range(start, len(ann)):
        if ann.at[i, "true_topic"] == "": return i
    for i in range(0, start):
        if ann.at[i, "true_topic"] == "": return i
    return None

pos = {"i": next_unlabelled() if next_unlabelled() is not None else len(ann) - 1}
progress, sentence, current = w.HTML(), w.HTML(), w.HTML()
buttons = [w.Button(description=t, layout=w.Layout(width="120px")) for t in TOPICS]
back = w.Button(description="◀ back", layout=w.Layout(width="90px"))
fwd = w.Button(description="next ▶", layout=w.Layout(width="90px"))
download = w.Button(description="download " + OUT, button_style="success", layout=w.Layout(width="250px"))

def render():
    i = pos["i"]; row = ann.iloc[i]
    n_done = (ann.true_topic != "").sum()
    progress.value = f"<b>Annotator {ANNOTATOR}</b> — sentence {i + 1} / {len(ann)} · {n_done} labelled"
    label = f"<br><span style='color:gray'>article topic: {row.inherited_label}</span>" if SHOW_LABEL else ""
    sentence.value = f"<div style='font-size:17px;max-width:800px;padding:10px 0'>{row.sentence}{label}</div>"
    current.value = (f"current answer: <b>{row.true_topic}</b>" if row.true_topic
                     else "<i>not labelled yet</i>")
    for b in buttons: b.button_style = "info" if b.description == row.true_topic else ""
    if n_done == len(ann):
        progress.value += " · <b style='color:green'>all done – click download</b>"

def choose(b):
    i = pos["i"]
    ann.at[i, "true_topic"] = b.description
    ann.at[i, "on_topic"] = "Y" if b.description == ann.at[i, "inherited_label"] else "N"
    ann.to_csv(OUT, index=False)                       # saved after every click
    nxt = next_unlabelled(i + 1)
    pos["i"] = nxt if nxt is not None else i
    render()

def move(step):
    pos["i"] = min(max(pos["i"] + step, 0), len(ann) - 1); render()

for b in buttons: b.on_click(choose)
back.on_click(lambda _: move(-1)); fwd.on_click(lambda _: move(1))
download.on_click(lambda _: files.download(OUT))
render()
display(w.VBox([progress, sentence, w.HBox(buttons), current, w.HBox([back, fwd]), download]))


In [ ]:
def train_e(inp, out, ng, lr, ep):
    t = time.time()
    subprocess.run(f"./fasttext supervised -input {inp} -output {out} -dim 10 -epoch {ep} "
                   f"-lr {lr} -wordNgrams {ng} -bucket 10000000 -minCount 1 "
                   f"-thread {THREADS} -verbose 0", shell=True, check=True)
    return time.time() - t

def tune_small(name, label):
    rows = open(f"{name}.train").read().splitlines()
    random.seed(0); random.shuffle(rows); cut = int(len(rows) * 0.9)
    open(f"{name}.trsub", "w").write("\n".join(rows[:cut]) + "\n")
    open(f"{name}.valid", "w").write("\n".join(rows[cut:]) + "\n")
    out = []
    for ng in (1, 2):
        # the paper's fixed recipe, for comparison
        train_e(f"{name}.train", "tmp", ng, 0.25, 5)
        out.append({"data": label, "bigrams": ng == 2, "setting": "paper (5 ep, lr 0.25)",
                    "acc_%": round(100 * acc("tmp", f"{name}.test"), 1)})
        # tuned on validation
        val = {}
        for ep in (5, 25, 50):
            for lr in (0.25, 0.5, 1.0):
                train_e(f"{name}.trsub", "tmp", ng, lr, ep)
                val[(ep, lr)] = acc("tmp", f"{name}.valid")
        ep, lr = max(val, key=val.get)
        secs = train_e(f"{name}.train", f"{name}_tuned_{ng}", ng, lr, ep)
        out.append({"data": label, "bigrams": ng == 2, "setting": f"tuned ({ep} ep, lr {lr})",
                    "acc_%": round(100 * acc(f"{name}_tuned_{ng}", f"{name}.test"), 1),
                    "train_sec": round(secs, 1)})
    return out

bbc2 = pd.DataFrame(tune_small("bbc", "BBC articles") + tune_small("bbcsent", "BBC sentences"))
display(bbc2); bbc2.to_csv("bbc_results_v2.csv", index=False); files.download("bbc_results_v2.csv")

## 7. LLM-as-judge
An LLM labels the same 100 sentences as a third, automatic annotator, using the **same questions** as the humans, plus one more:
was fastText's prediction a *plausible* mistake for this sentence alone (e.g. a tech-firm merger called business), or a clearly wrong one?

Uses Colab's free built-in model (`google.colab.ai`), so no API key is needed. The model name is printed and saved with every row.
That helper has no temperature setting, so answers may vary between runs: `llm_judge.csv` keeps every raw response as the record.
If the cell stops part-way (rate limits), just re-run it: rows already judged are kept and only the rest are asked.

In [ ]:
# ---- LLM-as-judge using Colab's free built-in model (no API key needed) ----
# self-contained: works even after a runtime restart (no need to re-run cells 1-6)
import json, os, re, time
import pandas as pd
from google.colab import ai, files
os.makedirs("/content/fastText", exist_ok=True); os.chdir("/content/fastText")
if not os.path.exists("bbc_annotation_key.csv"):
    print("bbc_annotation_key.csv not found - please upload it (and llm_judge.csv too, if you have a partial run)")
    files.upload()

print("Available models:", ai.list_models())
JUDGE_MODEL = ai.list_models()[0]          # change this to pick a different model; it is saved with the results
TOPICS = ["business", "entertainment", "politics", "sport", "tech"]

PROMPT = """You are helping evaluate a news-topic classifier on single sentences taken from BBC News articles.

Sentence: "{sentence}"
The article this sentence came from is labelled: {gold}
The classifier predicted: {pred}

Judge the sentence ON ITS OWN, without guessing the rest of the article. Reply with JSON only, using exactly these keys:
"true_topic": the topic this sentence alone most clearly signals, one of {topics}, or "none" if it signals no topic,
"on_topic": "Y" if this sentence alone signals the article label ({gold}), otherwise "N",
"pred_plausible": "Y" if a careful human reading only this sentence could reasonably choose the classifier's prediction ({pred}), otherwise "N",
"reason": one short sentence explaining your judgement."""

def ask(prompt):
    try:
        return ai.generate_text(prompt, model_name=JUDGE_MODEL)
    except TypeError:                       # older versions of the helper take no model argument
        return ai.generate_text(prompt)

def judge(row, tries=3):
    prompt = PROMPT.format(sentence=row.sentence, gold=row.inherited_label,
                           pred=row.fasttext_pred, topics=", ".join(TOPICS))
    for t in range(tries):
        try:
            raw = ask(prompt)
            out = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
            return {"llm_topic": str(out["true_topic"]).strip().lower(),
                    "llm_on_topic": str(out["on_topic"]).strip().upper()[:1],
                    "llm_pred_plausible": str(out["pred_plausible"]).strip().upper()[:1],
                    "llm_reason": out.get("reason", ""), "llm_raw": raw, "llm_model": str(JUDGE_MODEL)}
        except Exception as e:
            print(f"id {row.id}: attempt {t+1} failed ({e})"); time.sleep(5 * (t + 1))
    return {"llm_topic": "", "llm_on_topic": "", "llm_pred_plausible": "", "llm_reason": "",
            "llm_raw": "FAILED", "llm_model": str(JUDGE_MODEL)}

key = pd.read_csv("bbc_annotation_key.csv")

# resume: keep rows already judged in an earlier run, only ask about the rest
done = {}
if os.path.exists("llm_judge.csv"):
    prev = pd.read_csv("llm_judge.csv").fillna("")
    done = {r["id"]: r for r in prev.to_dict("records") if r.get("llm_raw") not in ("", "FAILED")}
results = []
for r in key.itertuples():
    if r.id in done:
        results.append({k: done[r.id][k] for k in ("llm_topic", "llm_on_topic", "llm_pred_plausible",
                                                   "llm_reason", "llm_raw", "llm_model")})
    else:
        results.append(judge(r))
        pd.concat([key.iloc[:len(results)], pd.DataFrame(results)], axis=1).to_csv("llm_judge.csv", index=False)
        if len(results) % 10 == 0: print(f"{len(results)} / {len(key)} judged")

judged = pd.concat([key, pd.DataFrame(results)], axis=1)
judged.to_csv("llm_judge.csv", index=False)

failed = (judged.llm_raw == "FAILED").sum()
if failed: print(f"{failed} sentences failed - just re-run this cell to retry only those")
errors = judged[judged.inherited_label != judged.fasttext_pred]
print(f"Judge model: {JUDGE_MODEL}")
print(f"LLM says on-topic: {(judged.llm_on_topic == 'Y').sum()} / 100")
print(f"fastText errors: {len(errors)}; judged plausible: {(errors.llm_pred_plausible == 'Y').sum()}")
print(f"fastText accuracy vs inherited label: {(judged.inherited_label == judged.fasttext_pred).mean():.2f}")
print(f"... counting plausible errors as correct: "
      f"{((judged.inherited_label == judged.fasttext_pred) | (judged.llm_pred_plausible == 'Y')).mean():.2f}")
display(errors[["sentence", "inherited_label", "fasttext_pred", "llm_topic", "llm_pred_plausible", "llm_reason"]])
files.download("llm_judge.csv")


## 8. Agreement and label-noise results
Upload `annotator_A.csv` and `annotator_B.csv` (filled-in copies of the sheet). If you only have one annotator, upload just A:
agreement is then reported between A and the LLM judge only.
Run section 7 first if you want the LLM included.

In [ ]:
# ---- Agreement (Cohen's kappa) and label-noise results ----
# Self-contained: works after a runtime restart.
import os
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from google.colab import files

os.makedirs("/content/fastText", exist_ok=True); os.chdir("/content/fastText")
TOPICS = ["business", "entertainment", "politics", "sport", "tech", "none"]
need = [f for f in ("bbc_annotation_key.csv", "annotator_A.csv") if not os.path.exists(f)]
if need:
    print("Please upload:", ", ".join(need), "(plus annotator_B.csv and llm_judge.csv if you have them)")
    files.upload()

read = lambda fn: pd.read_csv(fn, dtype=str, keep_default_na=False)      # everything as text, blanks stay ""
key = read("llm_judge.csv") if os.path.exists("llm_judge.csv") else read("bbc_annotation_key.csv")
key["id"] = key["id"].astype(int)

def clean(df, on_col, topic_col):
    return pd.DataFrame({"on_topic": df[on_col].str.strip().str.upper().str[:1],
                         "true_topic": df[topic_col].str.strip().str.lower()}).reset_index(drop=True)

raters = {}
for who in ("A", "B"):
    fn = f"annotator_{who}.csv"
    if os.path.exists(fn):
        d = read(fn); d["id"] = d["id"].astype(int)
        raters[who] = clean(d.set_index("id").reindex(key.id).fillna(""), "on_topic", "true_topic")
        print(f"annotator {who}: {(raters[who].true_topic != '').sum()} / {len(key)} sentences labelled")
if "llm_on_topic" in key:
    raters["LLM"] = clean(key, "llm_on_topic", "llm_topic")
    print(f"LLM judge: {(raters['LLM'].true_topic != '').sum()} / {len(key)} sentences judged")

rows, names = [], list(raters)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = raters[names[i]], raters[names[j]]
        for col, valid in (("on_topic", ["Y", "N"]), ("true_topic", TOPICS)):
            ok = a[col].isin(valid) & b[col].isin(valid)        # only rows both raters actually labelled
            rows.append({"pair": f"{names[i]} vs {names[j]}", "question": col, "n": int(ok.sum()),
                         "raw_agreement": round((a[col][ok] == b[col][ok]).mean(), 2),
                         "cohen_kappa": round(cohen_kappa_score(a[col][ok], b[col][ok]), 2)})
agree = pd.DataFrame(rows); display(agree)

# label noise and fastText accuracy, with annotator A as the reference
ft_correct = (key.inherited_label == key.fasttext_pred).reset_index(drop=True)
A = raters["A"]
on, off = A.on_topic == "Y", A.on_topic == "N"
print(f"\nAnnotator A: {on.sum()} / {on.sum() + off.sum()} sentences on-topic "
      f"-> estimated label noise {100 * off.sum() / (on.sum() + off.sum()):.0f}%")
print(f"  of the off-topic ones: {(A.true_topic[off] == 'none').sum()} signal no topic, "
      f"{(off & (A.true_topic != 'none')).sum()} signal a different topic")
print(f"fastText accuracy vs inherited label:      {ft_correct.mean():.2f} (n={len(key)})")
print(f"fastText accuracy on on-topic sentences:   {ft_correct[on].mean():.2f} (n={on.sum()})")
print(f"fastText accuracy on off-topic sentences:  {ft_correct[off].mean():.2f} (n={off.sum()})")
has_topic = A.true_topic.isin(TOPICS[:-1])
print(f"fastText accuracy vs A's sentence topic:   "
      f"{(key.fasttext_pred.reset_index(drop=True)[has_topic] == A.true_topic[has_topic]).mean():.2f} (n={has_topic.sum()})")
agree.to_csv("agreement.csv", index=False); files.download("agreement.csv")


In [ ]:
!ls -la /content/fastText/*.csv
display(pd.read_csv("replication.csv"))
files.download("replication.csv")